# PanTS pancreatic-tumor segmentation with nnU-Net v2

This notebook is the executable, reproducible runbook for the PanTS nnU-Net baseline. It uses project code only for PanTS-specific data validation/conversion and uses the official nnU-Net v2 CLI for fingerprinting, planning, preprocessing, training, inference, evaluation, checkpointing, and model export.

## Scientific target

- Input: one three-dimensional abdominal CT channel (`0000`).
- Target: one exclusive semantic label map containing background 0 and PanTS foreground labels 1-28.
- Primary endpoint: pancreatic-lesion detection/localization/segmentation (label 28).
- Auxiliary supervision: pancreas subdivisions, duct, vessels, and surrounding organs.

## Two experiment tiers

| Tier | Dataset | Purpose | Permitted claim |
|---|---|---|---|
| Engineering smoke | Dataset501, 40 PanTS-tr cases | Prove the complete pipeline | Integration only; never benchmark performance |
| Production baseline | Dataset500, all 9,000 PanTS-tr cases | Five-fold model development | Internal cross-validation, then one locked PanTS-te evaluation |

This notebook executes Dataset501 by default. Dataset500 requires persistent HPC/cloud storage and is documented at the end; it is not silently launched on Colab.

## Data and leakage policy

`PanTS/` is immutable. PanTS-tr alone is used for training, model selection, thresholds, and postprocessing. PanTS-te is not used until the model and evaluation protocol are frozen. The public data do not expose a patient-group identifier, so case-level cross-validation cannot prove patient independence; this limitation must be reported or resolved with the dataset authors.

## Colab limitation

Managed Colab runtimes are ephemeral and cannot be guaranteed not to disconnect. Active arrays live under `/content`; durable archives, checkpoints, run records, and model exports live in Google Drive. Do not use keep-alive workarounds.

## 0. Runtime and download gate

Do not change runtime while a PanTS download cell is active: switching runtimes terminates the current VM. Complete both official download scripts and verify their exit status first.

Recommended staged use:

1. CPU + High-RAM: download verification, Dataset501 creation, fingerprinting, planning, preprocessing, and archive creation.
2. A100 GPU + High-RAM: restore the preprocessing archive, train, validate, predict, evaluate, and export.

High-RAM is host memory, not GPU VRAM. Colab does not guarantee an A100 variant, runtime duration, local-disk size, or availability; inspect the actual allocation every session. The notebook refuses PyTorch 2.9 because nnU-Net 2.8.1 explicitly excludes that release series.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import shlex
import shutil
import subprocess
import sys

def run(command: list[str], cwd: Path | None = None) -> None:
    """Display and run one command; stop the notebook on failure."""
    command = [str(item) for item in command]
    print('$', shlex.join(command))
    subprocess.run(
        command,
        cwd=None if cwd is None else str(cwd),
        check=True,
        env=os.environ.copy(),
    )

def sha256_file(path: Path) -> str:
    """Return a streaming SHA-256 checksum without loading a file into RAM."""
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

PANTS_DRIVE = Path('/content/drive/MyDrive/PanTS')
PANTS_DATA = PANTS_DRIVE / 'data'
ARTIFACTS = Path('/content/drive/MyDrive/PanTS_nnUNet_artifacts')
ARTIFACTS.mkdir(parents=True, exist_ok=True)

if shutil.which('nvidia-smi'):
    run([
        'nvidia-smi',
        '--query-gpu=name,memory.total,memory.used,memory.free',
        '--format=csv,noheader',
    ])
else:
    print('No GPU is attached; this is acceptable before training.')

run(['free', '-h'])
run(['df', '-h', '/content', '/content/drive'])
run(['nproc'])

## 1. Verify the immutable PanTS download

This is a completeness audit, not model development on PanTS-te. Directory-name equality proves that each image case has a corresponding label case and that train/test identifiers do not overlap. This cell should pass before any nnU-Net conversion.

In [ ]:
def case_ids(folder: Path) -> list[str]:
    if not folder.is_dir():
        raise FileNotFoundError(folder)
    return sorted(path.name for path in folder.iterdir() if path.is_dir())

train_images = case_ids(PANTS_DATA / 'ImageTr')
train_labels = case_ids(PANTS_DATA / 'LabelTr')
test_images = case_ids(PANTS_DATA / 'ImageTe')
test_labels = case_ids(PANTS_DATA / 'LabelTe')

print('ImageTr:', len(train_images))
print('LabelTr:', len(train_labels))
print('ImageTe:', len(test_images))
print('LabelTe:', len(test_labels))

assert len(train_images) == len(train_labels) == 9000
assert train_images == train_labels
assert len(test_images) == len(test_labels) == 901
assert test_images == test_labels
assert set(train_images).isdisjoint(test_images)

for case_id in (train_images[0], train_images[-1]):
    assert (PANTS_DATA / 'ImageTr' / case_id / 'ct.nii.gz').is_file()
    assert (PANTS_DATA / 'LabelTr' / case_id / 'combined_labels.nii.gz').is_file()

print('PanTS image/label case pairing passed.')

## 2. Clone the version-controlled PanTS adapter

The notebook does not duplicate the data adapter. It checks out the immutable `nnunet-colab-v1` Git tag, then records the resolved commit SHA. The tag removes the need for a fragile hard-coded commit placeholder while still pinning one exact source revision. Standard nnU-Net remains an installed dependency; its source is not copied into this repository.

In [ ]:
PROJECT_REPOSITORY = 'https://github.com/sabinthapa100/pants_sabin.git'
PROJECT_REF = 'nnunet-colab-v1'
PROJECT_DIR = Path('/content/pants_sabin')

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / '.git').is_dir():
        raise RuntimeError(f'Unexpected non-Git path: {PROJECT_DIR}')
else:
    run([
        'git', 'clone',
        '--depth', '1',
        '--branch', PROJECT_REF,
        PROJECT_REPOSITORY,
        str(PROJECT_DIR),
    ])

PROJECT_COMMIT = subprocess.check_output(
    ['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'],
    text=True,
).strip()
TAG_COMMIT = subprocess.check_output(
    ['git', '-C', str(PROJECT_DIR), 'rev-list', '-n', '1', PROJECT_REF],
    text=True,
).strip()
assert PROJECT_COMMIT == TAG_COMMIT
print('Pinned project commit:', PROJECT_COMMIT)
(ARTIFACTS / 'project_commit.txt').write_text(
    PROJECT_COMMIT + '\n', encoding='utf-8'
)

## 3. Verify PyTorch, then install the pinned Python dependencies

PyTorch is hardware-specific and is therefore supplied by the selected Colab runtime rather than the repository requirements. We inspect it before installing nnU-Net. `pip check` detects dependency conflicts, and the environment record makes the run auditable.

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('PyTorch CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

torch_base_version = torch.__version__.split('+')[0]
if torch_base_version.startswith('2.9.'):
    raise RuntimeError(
        'nnunetv2 2.8.1 excludes PyTorch 2.9. Select another Colab runtime.'
    )

run([
    sys.executable, '-m', 'pip', 'install',
    '-r', str(PROJECT_DIR / 'requirements.txt'),
])
run([sys.executable, '-m', 'pip', 'check'])

from importlib.metadata import version
import nnunetv2
import torchvision

assert version('nnunetv2') == '2.8.1'
print('nnunetv2:', version('nnunetv2'))
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)

environment_record = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'project_commit': PROJECT_COMMIT,
    'python': sys.version,
    'torch': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'torchvision': torchvision.__version__,
    'nnunetv2': version('nnunetv2'),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(ARTIFACTS / 'environment.json').write_text(
    json.dumps(environment_record, indent=2) + '\n',
    encoding='utf-8',
)
print(json.dumps(environment_record, indent=2))

## 4. Create and verify Dataset501

A runtime-only link makes the immutable Drive PanTS tree visible at the path expected by the repository. `prepare_nnunet.py` then selects a deterministic 40-case PanTS-tr cohort, validates integer labels and CT/label geometry, creates 40 image and 40 label links, and writes `NibabelIOWithReorient` into `dataset.json`. It does not preprocess or modify PanTS.

The raw and preprocessed nnU-Net workspaces remain under fast ephemeral `/content`. Smoke checkpoints/results are directed to Drive for durability.

In [ ]:
pants_link = PROJECT_DIR / 'PanTS'
if pants_link.is_symlink():
    assert pants_link.resolve() == PANTS_DRIVE.resolve()
elif pants_link.exists():
    raise RuntimeError(f'Unexpected path already exists: {pants_link}')
else:
    pants_link.symlink_to(PANTS_DRIVE, target_is_directory=True)

NNUNET_ROOT = PROJECT_DIR / 'nnunet'
NNUNET_RAW = NNUNET_ROOT / 'nnUNet_raw'
NNUNET_PREPROCESSED = NNUNET_ROOT / 'nnUNet_preprocessed'
NNUNET_RESULTS = ARTIFACTS / 'nnUNet_results'
DATASET_NAME = 'Dataset501_PanTSSmoke'
DATASET_DIR = NNUNET_RAW / DATASET_NAME

if not DATASET_DIR.exists():
    run([
        sys.executable,
        'scripts/prepare_nnunet.py',
        '--dataset-id', '501',
        '--name', 'PanTSSmoke',
        '--max-cases', '40',
    ], cwd=PROJECT_DIR)

for directory in (NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS):
    directory.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)
os.environ['nnUNet_def_n_proc'] = '1'
os.environ['nnUNet_n_proc_DA'] = '0'
os.environ['nnUNet_compile'] = 'False'

dataset_json_path = DATASET_DIR / 'dataset.json'
dataset_json = json.loads(dataset_json_path.read_text())
images = sorted((DATASET_DIR / 'imagesTr').glob('*.nii.gz'))
labels = sorted((DATASET_DIR / 'labelsTr').glob('*.nii.gz'))

assert len(images) == len(labels) == dataset_json['numTraining'] == 40
assert all(path.is_symlink() and path.exists() for path in images + labels)
assert not (DATASET_DIR / 'imagesTs').exists()
assert dataset_json['channel_names'] == {'0': 'CT'}
assert dataset_json['labels']['pancreatic_lesion'] == 28
assert dataset_json['overwrite_image_reader_writer'] == 'NibabelIOWithReorient'

print(json.dumps(dataset_json, indent=2))
print('Dataset501 validation passed:', DATASET_DIR)

## 5. Reuse the numerical and visual QC tools

This is a pre-training sanity check. The CT display window is visualization only and is not nnU-Net preprocessing. PanTS_00000003 is a verified lesion-positive PanTS-tr case.

In [ ]:
run([
    sys.executable, 'scripts/inspect_data.py',
    '--case', 'PanTS_00000003',
], cwd=PROJECT_DIR)
run([
    sys.executable, 'scripts/visualize_data.py',
    '--case', 'PanTS_00000003',
    '--structures', 'pancreas', 'pancreatic_lesion',
], cwd=PROJECT_DIR)

from IPython.display import Image, display
display(Image(filename=str(
    PROJECT_DIR / 'outputs/figures/PanTS_00000003_lesion.png'
)))

## 6. Integrity checking, fingerprinting, and default planning

The fingerprint summarizes training-set spacings, shapes, crop behavior, and foreground CT intensity statistics. The planner converts these empirical properties into target spacing, patch size, batch size, normalization, resampling, and network topology. `--no_pp` deliberately stops before preprocessing, and `-npfp 1` limits host-memory pressure.

We first establish the default `nnUNetPlans` reference. We do not alter target spacing, patch size, batch size, planner, or GPU-memory target.

In [ ]:
PREPROCESSED_DATASET = NNUNET_PREPROCESSED / DATASET_NAME
fingerprint_path = PREPROCESSED_DATASET / 'dataset_fingerprint.json'
plans_path = PREPROCESSED_DATASET / 'nnUNetPlans.json'

if not fingerprint_path.is_file() or not plans_path.is_file():
    run([
        'nnUNetv2_plan_and_preprocess',
        '-d', '501',
        '--verify_dataset_integrity',
        '--no_pp',
        '-npfp', '1',
    ])

fingerprint = json.loads(fingerprint_path.read_text())
plans = json.loads(plans_path.read_text())
configuration = plans['configurations']['3d_fullres']

assert plans['image_reader_writer'] == 'NibabelIOWithReorient'
print('Number of fingerprint cases:', len(fingerprint['spacings']))
print('Image reader/writer:', plans['image_reader_writer'])
print('Transpose forward/backward:', plans['transpose_forward'], plans['transpose_backward'])
print('Available configurations:', list(plans['configurations']))
print('3d_fullres spacing:', configuration['spacing'])
print('3d_fullres patch:', configuration['patch_size'])
print('3d_fullres batch:', configuration['batch_size'])
print('Normalization:', configuration['normalization_schemes'])
print('Architecture:', json.dumps(configuration['architecture'], indent=2))

## 7. Preprocess only corrected 3D full resolution

Preprocessing reads the raw NIfTIs with `NibabelIOWithReorient`, crops if applicable, applies CT normalization learned from PanTS-tr, resamples CT intensities and discrete segmentations appropriately, and writes memory-mappable `.b2nd` arrays plus `.pkl` case properties. It does not alter PanTS.

`3d_lowres` is not chosen merely because of its name: physical resolution and peak training VRAM are different quantities. One worker is used to avoid the host-RAM exhaustion previously seen locally.

In [ ]:
configuration_folder = PREPROCESSED_DATASET / 'nnUNetPlans_3d_fullres'
if configuration_folder.exists():
    raise FileExistsError(
        f'{configuration_folder} already exists; refusing an ambiguous rerun.'
    )

run([
    'nnUNetv2_preprocess',
    '-d', '501',
    '-c', '3d_fullres',
    '-np', '1',
])

data_files = [
    path for path in configuration_folder.glob('*.b2nd')
    if not path.name.endswith('_seg.b2nd')
]
segmentation_files = list(configuration_folder.glob('*_seg.b2nd'))
property_files = list(configuration_folder.glob('*.pkl'))
ground_truth_files = list((PREPROCESSED_DATASET / 'gt_segmentations').glob('*.nii.gz'))

assert len(data_files) == 40
assert len(segmentation_files) == 40
assert len(property_files) == 40
assert len(ground_truth_files) == 40

print('Preprocessed images:', len(data_files))
print('Preprocessed segmentations:', len(segmentation_files))
print('Properties:', len(property_files))
run(['du', '-sh', str(PREPROCESSED_DATASET)])

## 8. Freeze a lesion-stratified five-fold split

The smoke cohort is deliberately balanced 20/20. A fixed stratified split gives every fold four lesion-positive and four lesion-negative validation cases. Each fold therefore trains on 32 cases and validates on eight. This split is written in nnU-Net's official `splits_final.json` format and must be reused for any fair architecture comparison on Dataset501.

This is still case-level, not proven patient-level, splitting because PanTS does not provide a public patient-group key.

In [ ]:
import nibabel as nib
import numpy as np
from sklearn.model_selection import StratifiedKFold

case_names = sorted(
    path.name.removesuffix('.nii.gz')
    for path in (DATASET_DIR / 'labelsTr').glob('*.nii.gz')
)
lesion_status = []
for case_id in case_names:
    label_image = nib.load(str(DATASET_DIR / 'labelsTr' / f'{case_id}.nii.gz'))
    lesion_status.append(int(np.any(np.asanyarray(label_image.dataobj) == 28)))

splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=12345)
splits = []
for train_indices, val_indices in splitter.split(case_names, lesion_status):
    splits.append({
        'train': [case_names[index] for index in train_indices],
        'val': [case_names[index] for index in val_indices],
    })

splits_path = PREPROCESSED_DATASET / 'splits_final.json'
if splits_path.exists():
    assert json.loads(splits_path.read_text()) == splits
else:
    splits_path.write_text(
        json.dumps(splits, indent=2) + '\n', encoding='utf-8'
    )

status_by_case = dict(zip(case_names, lesion_status))
for fold, split in enumerate(splits):
    train_positive = sum(status_by_case[case_id] for case_id in split['train'])
    val_positive = sum(status_by_case[case_id] for case_id in split['val'])
    print(
        f'fold {fold}: train={len(split["train"])} '
        f'(positive={train_positive}), val={len(split["val"])} '
        f'(positive={val_positive})'
    )

## 9. Archive preprocessing before changing runtime

The preprocessing directory is expensive but reproducible. A single uncompressed tar reduces Drive small-file operations; SHA-256 detects incomplete or corrupted transfer. After this cell succeeds, save the notebook and switch to A100 + High-RAM. Switching deletes `/content`.

In [ ]:
archive_local = Path('/content/Dataset501_PanTSSmoke_preprocessed.tar')
archive_drive = ARTIFACTS / archive_local.name
checksum_drive = ARTIFACTS / f'{archive_local.name}.sha256'

if archive_drive.exists() or checksum_drive.exists():
    raise FileExistsError(
        'A preprocessing archive already exists. Verify it instead of overwriting it.'
    )

run([
    'tar', '-C', str(NNUNET_PREPROCESSED),
    '-cf', str(archive_local), DATASET_NAME,
])
archive_digest = sha256_file(archive_local)
shutil.copy2(archive_local, archive_drive)
checksum_drive.write_text(
    f'{archive_digest}  {archive_drive.name}\n', encoding='utf-8'
)
assert sha256_file(archive_drive) == archive_digest
print('Durable preprocessing archive:', archive_drive)
print('SHA-256:', archive_digest)

## 10. A100-session restore path

After selecting A100 + High-RAM, rerun sections 0-4 to remount Drive, clone/install the pinned code, recreate the inexpensive Dataset501 raw links, and reset the environment variables. Then run this restore cell instead of repeating fingerprinting/preprocessing.

In [ ]:
archive_drive = ARTIFACTS / 'Dataset501_PanTSSmoke_preprocessed.tar'
checksum_drive = ARTIFACTS / 'Dataset501_PanTSSmoke_preprocessed.tar.sha256'
expected_digest = checksum_drive.read_text().split()[0]
assert sha256_file(archive_drive) == expected_digest

restored_dataset = NNUNET_PREPROCESSED / DATASET_NAME
if not restored_dataset.exists():
    archive_local = Path('/content') / archive_drive.name
    shutil.copy2(archive_drive, archive_local)
    assert sha256_file(archive_local) == expected_digest
    run([
        'tar', '-C', str(NNUNET_PREPROCESSED),
        '-xf', str(archive_local),
    ])

PREPROCESSED_DATASET = restored_dataset
splits_path = PREPROCESSED_DATASET / 'splits_final.json'
plans_path = PREPROCESSED_DATASET / 'nnUNetPlans.json'
assert splits_path.is_file()
assert plans_path.is_file()
assert (PREPROCESSED_DATASET / 'nnUNetPlans_3d_fullres').is_dir()
print('Preprocessed Dataset501 restored and verified.')

## 11. Controlled fold-0 smoke training and checkpoint recovery

`nnUNetTrainer_1epoch` is an official debug trainer. One epoch still contains 250 optimizer iterations and 50 online patch-validation iterations. The initial action is `fresh` and deliberately omits `--val`; normal training automatically performs full-volume validation after training.

Actions:

- `fresh`: start a new fold only if its result directory does not exist.
- `resume`: add `--c` and continue the same trainer/plans/configuration/fold from an available checkpoint.
- `validate_only`: add `--val`; this performs no optimizer steps and requires completed training.

Results are written to Drive for smoke-run durability. Do not use `--disable_checkpointing`. `--npz` is omitted because 29-class probability volumes are large and are unnecessary for this single-configuration integration test.

In [ ]:
assert torch.cuda.is_available(), 'Attach a GPU before training.'
run([
    'nvidia-smi',
    '--query-gpu=name,memory.total,memory.used,memory.free',
    '--format=csv,noheader',
])
run(['free', '-h'])

CONFIGURATION = '3d_fullres'
FOLD = 0
TRAINER = 'nnUNetTrainer_1epoch'
PLANS = 'nnUNetPlans'
ACTION = 'fresh'  # change deliberately to: resume or validate_only

result_fold = (
    NNUNET_RESULTS / DATASET_NAME
    / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
    / f'fold_{FOLD}'
)
command = [
    'nnUNetv2_train', '501', CONFIGURATION, str(FOLD),
    '-tr', TRAINER, '-p', PLANS,
]

if ACTION == 'fresh':
    if result_fold.exists():
        raise FileExistsError(
            f'{result_fold} exists; choose resume or validate_only deliberately.'
        )
elif ACTION == 'resume':
    command.append('--c')
elif ACTION == 'validate_only':
    command.append('--val')
else:
    raise ValueError(f'Unknown ACTION: {ACTION}')

run(command)

## 12. Require completed whole-case validation

The proper smoke completion artifact is `validation/summary.json` plus one prediction for every fold-0 validation case. `progress.png` and online pseudo-Dice summarize sampled validation patches and are not substitutes for whole-volume evaluation.

The all-foreground macro mean averages 28 foreground structures and is not tumor Dice. Pancreatic-lesion performance is label 28.

In [ ]:
validation_folder = result_fold / 'validation'
validation_summary_path = validation_folder / 'summary.json'
if not validation_summary_path.is_file():
    raise FileNotFoundError(
        'Full-volume validation is incomplete. Diagnose before reporting metrics.'
    )

splits = json.loads(splits_path.read_text())
expected_validation_ids = set(splits[FOLD]['val'])
predicted_validation_ids = {
    path.name.removesuffix('.nii.gz')
    for path in validation_folder.glob('PanTS_*.nii.gz')
}
assert predicted_validation_ids == expected_validation_ids

validation_summary = json.loads(validation_summary_path.read_text())
tumor_metrics = validation_summary['mean'].get('28')
if tumor_metrics is None:
    raise KeyError(f'Label 28 missing; keys={validation_summary["mean"].keys()}')

print('Whole-case validation cases:', sorted(predicted_validation_ids))
print('Label-28 pancreatic-lesion metrics:')
print(json.dumps(tumor_metrics, indent=2))
print('All-foreground macro mean (not tumor performance):')
print(json.dumps(validation_summary['foreground_mean'], indent=2))

## 13. Test the raw-input prediction and evaluation interfaces

This repeats inference on the same fold-0 held-out cases, but starts from raw `_0000.nii.gz` inputs. It therefore verifies preprocessing-at-inference, sliding-window prediction, export to original geometry, and independent folder evaluation. It is an interface test, not a second independent test set.

Only fold 0 was trained, so `-f 0` is mandatory. Default inference would expect folds 0-4. If full-volume accumulation exceeds GPU VRAM, set `ACCUMULATE_ON_CPU=True`; `--not_on_device` then moves the accumulator, not the neural network, to host RAM.

In [ ]:
import tempfile

interface_root = Path(tempfile.mkdtemp(prefix='pants_fold0_interface_'))
inference_input = interface_root / 'images'
inference_ground_truth = interface_root / 'ground_truth'
inference_predictions = interface_root / 'predictions'
for directory in (inference_input, inference_ground_truth, inference_predictions):
    directory.mkdir()

for case_id in sorted(expected_validation_ids):
    (inference_input / f'{case_id}_0000.nii.gz').symlink_to(
        DATASET_DIR / 'imagesTr' / f'{case_id}_0000.nii.gz'
    )
    (inference_ground_truth / f'{case_id}.nii.gz').symlink_to(
        DATASET_DIR / 'labelsTr' / f'{case_id}.nii.gz'
    )

ACCUMULATE_ON_CPU = False
predict_command = [
    'nnUNetv2_predict',
    '-i', str(inference_input),
    '-o', str(inference_predictions),
    '-d', '501',
    '-c', CONFIGURATION,
    '-p', PLANS,
    '-tr', TRAINER,
    '-f', str(FOLD),
    '-chk', 'checkpoint_final.pth',
    '-npp', '1',
    '-nps', '1',
]
if ACCUMULATE_ON_CPU:
    predict_command.append('--not_on_device')
run(predict_command)

run([
    'nnUNetv2_evaluate_folder',
    str(inference_ground_truth),
    str(inference_predictions),
    '-djfile', str(dataset_json_path),
    '-pfile', str(plans_path),
    '-np', '1',
])

for case_id in sorted(expected_validation_ids):
    reference = nib.load(str(inference_ground_truth / f'{case_id}.nii.gz'))
    prediction = nib.load(str(inference_predictions / f'{case_id}.nii.gz'))
    assert reference.shape == prediction.shape
    assert np.allclose(reference.affine, prediction.affine, atol=1e-5)

interface_summary = json.loads(
    (inference_predictions / 'summary.json').read_text()
)
print('Raw-interface label-28 Dice:', interface_summary['mean']['28']['Dice'])
print('Prediction geometry matches the source labels for all fold-0 cases.')

## Evaluation boundary

The official nnU-Net evaluator reports voxel-wise Dice, IoU, TP, FP, FN, TN, and predicted/reference voxel counts. It does **not** implement the PanTS leaderboard's patient-wise sensitivity, tumor-wise sensitivity, specificity operating point, or AUC. The public PanTS material does not fully specify the required lesion matching, connected-component filtering, thresholding, continuous case score, and empty-case aggregation.

Therefore this notebook labels its outputs `INTERNAL nnU-Net semantic validation`. It does not invent or claim official PanTS P-Sen/T-Sen/Spe/AUC. A benchmark evaluator becomes legitimate only after JHU supplies the protocol or we predeclare a separately named internal protocol.

## 14. Save the run record and interface summary

A model filename is not an experiment record. This cell persists the exact source revision, software, dataset, split, plans, trainer, hardware, checkpoint, and metrics used by the run.

In [ ]:
RUN_ID = (
    f'Dataset501_{TRAINER}_{PLANS}_{CONFIGURATION}_fold{FOLD}'
)
RUN_ARTIFACTS = ARTIFACTS / 'runs' / RUN_ID
RUN_ARTIFACTS.mkdir(parents=True, exist_ok=True)

run_record = {
    'run_id': RUN_ID,
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'project_commit': PROJECT_COMMIT,
    'nnunetv2': version('nnunetv2'),
    'torch': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
    'dataset': DATASET_NAME,
    'num_training_cases': 40,
    'fold': FOLD,
    'fold_train_ids': splits[FOLD]['train'],
    'fold_validation_ids': splits[FOLD]['val'],
    'configuration': CONFIGURATION,
    'plans': PLANS,
    'trainer': TRAINER,
    'reader_writer': plans['image_reader_writer'],
    'target_spacing': configuration['spacing'],
    'patch_size': configuration['patch_size'],
    'batch_size': configuration['batch_size'],
    'checkpoint': 'checkpoint_final.pth',
    'label_28_validation': tumor_metrics,
    'label_28_raw_interface': interface_summary['mean']['28'],
    'scientific_status': 'engineering smoke test; not benchmark performance',
}
(RUN_ARTIFACTS / 'run_record.json').write_text(
    json.dumps(run_record, indent=2) + '\n', encoding='utf-8'
)
shutil.copy2(validation_summary_path, RUN_ARTIFACTS / 'validation_summary.json')
shutil.copy2(inference_predictions / 'summary.json', RUN_ARTIFACTS / 'interface_summary.json')
shutil.copy2(splits_path, RUN_ARTIFACTS / 'splits_final.json')
shutil.copy2(plans_path, RUN_ARTIFACTS / 'nnUNetPlans.json')
shutil.copy2(dataset_json_path, RUN_ARTIFACTS / 'dataset.json')
print(json.dumps(run_record, indent=2))

## 15. Export a portable model ZIP

The official exporter packages the checkpoint and nnU-Net metadata needed to reinstall or predict elsewhere. SHA-256 verifies transfer integrity. A one-epoch ZIP is a debug artifact, not a scientifically trained PanTS model.

In [ ]:
model_zip_local = Path('/content') / f'{RUN_ID}.zip'
model_zip_drive = ARTIFACTS / 'models' / model_zip_local.name
model_zip_drive.parent.mkdir(parents=True, exist_ok=True)
if model_zip_drive.exists():
    raise FileExistsError(model_zip_drive)

run([
    'nnUNetv2_export_model_to_zip',
    '-d', '501',
    '-o', str(model_zip_local),
    '-c', CONFIGURATION,
    '-tr', TRAINER,
    '-p', PLANS,
    '-f', str(FOLD),
    '-chk', 'checkpoint_final.pth',
])
model_digest = sha256_file(model_zip_local)
shutil.copy2(model_zip_local, model_zip_drive)
model_zip_drive.with_suffix('.zip.sha256').write_text(
    f'{model_digest}  {model_zip_drive.name}\n', encoding='utf-8'
)
assert sha256_file(model_zip_drive) == model_digest
print('Model ZIP:', model_zip_drive)
print('SHA-256:', model_digest)

## 16. Optional private Hugging Face upload

Create a fine-grained write token in Colab Secrets named `HF_TOKEN`; never paste it into notebook text. Upload only the model ZIP, checksum, and research documentation. Do not upload PanTS CTs, labels, metadata, reports, or preprocessed arrays. Keep the repository private until the PanTS authors clarify public redistribution of learned weights.

In [ ]:
PUSH_TO_HUB = False

if PUSH_TO_HUB:
    run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub'])
    from google.colab import userdata
    from huggingface_hub import HfApi

    token = userdata.get('HF_TOKEN')
    if not token:
        raise RuntimeError('HF_TOKEN is missing from Colab Secrets.')

    repo_id = 'REPLACE_WITH_YOUR_HF_USERNAME/pants-nnunet-smoke'
    api = HfApi(token=token)
    api.create_repo(repo_id=repo_id, repo_type='model', private=True, exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(model_zip_drive),
        path_in_repo=model_zip_drive.name,
        repo_id=repo_id,
        repo_type='model',
    )
    api.upload_file(
        path_or_fileobj=str(model_zip_drive.with_suffix('.zip.sha256')),
        path_in_repo=model_zip_drive.with_suffix('.zip.sha256').name,
        repo_id=repo_id,
        repo_type='model',
    )
    print('Private model repository:', repo_id)

# Production nnU-Net benchmark protocol — do not run from Dataset501

A postdoctoral-quality result requires a distinct Dataset500 containing all 9,000 PanTS-tr cases on persistent, high-throughput storage. Dataset membership controls training data; there is no scientifically sound trainer flag that turns a 40-case smoke dataset into the production cohort.

## Frozen development design

1. Prepare Dataset500 from PanTS-tr only with the same 29-class target and `NibabelIOWithReorient`.
2. Establish and freeze five folds. Stratify tumor status and use patient/source groups if the authors provide a valid grouping key. Reuse identical folds for every model.
3. Generate fingerprint/default plans and preprocess on NERSC `$PSCRATCH`.
4. Train the standard `nnUNetTrainer` (1,000 epochs, not the debug trainer) on folds 0-4.
5. Use only PanTS-tr out-of-fold predictions for model/configuration/postprocessing selection.
6. Freeze model, folds, checkpoint, sliding-window overlap, TTA, and evaluation rules.
7. Predict all 901 PanTS-te cases once using the five-fold ensemble.
8. Run generic semantic evaluation and the official PanTS evaluator only when its complete protocol is available.

Reference commands, one persistent job per fold:

```bash
nnUNetv2_train 500 3d_fullres 0 -tr nnUNetTrainer -p nnUNetPlans
nnUNetv2_train 500 3d_fullres 1 -tr nnUNetTrainer -p nnUNetPlans
nnUNetv2_train 500 3d_fullres 2 -tr nnUNetTrainer -p nnUNetPlans
nnUNetv2_train 500 3d_fullres 3 -tr nnUNetTrainer -p nnUNetPlans
nnUNetv2_train 500 3d_fullres 4 -tr nnUNetTrainer -p nnUNetPlans
```

Resume the exact same job identity with `--c`. Do not add `--npz` unless probability-level configuration ensembling is explicitly planned and storage for 29-class volumes has been budgeted. For one locked configuration, determine cross-validation postprocessing without configuration ensembling:

```bash
nnUNetv2_find_best_configuration 500 \
  -c 3d_fullres -p nnUNetPlans -tr nnUNetTrainer \
  -f 0 1 2 3 4 --disable_ensembling -np 1
```

After freezing development decisions, construct a flat PanTS-te input folder with `<CASE>_0000.nii.gz` CT names and run the five-fold ensemble:

```bash
nnUNetv2_predict \
  -i PANTS_TEST_INPUT -o PANTS_TEST_PREDICTIONS \
  -d 500 -c 3d_fullres -p nnUNetPlans -tr nnUNetTrainer \
  -f 0 1 2 3 4 -step_size 0.5 -npp 1 -nps 1
```

Keep default mirroring TTA; do not pass `--disable_tta` for the accuracy result. The modern ResEnc L preset is a separate experiment, not a silent replacement for this reference baseline. It requires substantially more VRAM/runtime and must be compared by the same out-of-fold protocol.

Full 9,000-case preprocessing and five-fold training are not reliable on ordinary Colab Pro because raw plus derived storage is hundreds of gigabytes and managed runtimes have finite lifetimes. Use NERSC or dedicated persistent cloud infrastructure.

# Primary references

- [nnU-Net repository and documentation](https://github.com/MIC-DKFZ/nnUNet)
- [nnU-Net dataset format](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/reference/dataset-format.md)
- [nnU-Net planning and preprocessing](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/how-to/plan-and-preprocess.md)
- [nnU-Net training](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/how-to/train-models.md)
- [nnU-Net inference](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/how-to/run-inference.md)
- [Residual encoder presets](https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md)
- [PanTS repository and benchmark description](https://github.com/MrGiovanni/PanTS)
- [Google Colab runtime/resource limitations](https://research.google.com/colaboratory/faq.html)

Cite Isensee et al., *Nature Methods* 2021 for nnU-Net and Li et al., NeurIPS 2025 for PanTS. If residual-encoder plans are later evaluated, also cite *nnU-Net Revisited* (2024).

# Smoke completion checklist

The notebook is complete only if all of the following are true:

- the pinned Git commit is recorded;
- Dataset501 contains exactly 40 paired cases and uses `NibabelIOWithReorient`;
- fingerprint and default plans exist;
- all 40 `3d_fullres` preprocessed image/label/property files exist;
- the frozen split has 32 train and 8 validation cases per fold;
- forward, loss, backward, and optimizer steps finish;
- `checkpoint_final.pth` exists;
- all eight fold-0 whole-volume predictions and `summary.json` exist;
- raw-input inference restores source geometry;
- label-28 semantic metrics are reported as internal smoke metrics only;
- the run record, model ZIP, and SHA-256 checksum are durable in Drive.

Passing this checklist demonstrates engineering correctness. It does not establish clinical validity or PanTS benchmark performance.